<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# **Operaciones de Aprendizaje Automático III**
# **Clase 5, vLLM**

Este colab responde 4 preguntas:

1. **por qué no alcanza con transformers?** Las dos bibliotecas generan lo mismo, se medirá cuánto tardan
2. **qué es la plantilla de chat?** Corre en CPU y puede ser causa de error
3. **qué significa servir un modelo?** Se levanta el servidor y se le pega con requests
4. **cómo se compone base + adaptador al desplegar?** Se entrena un adaptador mínimo y se cambia de modelo cambiando un nombre en el pedido.


---

**Instalamos lo necesario**

* vLLM como motor de inferencia
* peft para entrenar el adaptador LoRA
* torchao como dependencia de cuantización que vLLM arrastra

In [ ]:
!pip install "vllm==0.26.0" "peft>=0.14" "torchao>=0.17"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of nvidia-cudnn-frontend to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of quack-kernels to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of xgrammar to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.7/303.7 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 1.6 MB/s eta

---
## **Parte A) vLLM**

* transformers arma lotes de tamaño fijo: reserva el peor caso de longitud para todas las secuencias del lote y espera a que la más lenta termine
* vLLM maneja la caché de claves y valores en bloques, como la memoria virtual de un sistema operativo, y mete una secuencia
nueva en cuanto otra termina.

...

* las dos bibliotecas (transformers y vLLM) generan el mismo texto
* la diferencia está en qué hacen con la memoria mientras generan sesenta y cuatro respuestas a la vez

In [ ]:
%%writefile comparar.py
"""mismo modelo, mismos prompts, mismo largo, solo cambia quien genera"""

import gc
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO = "Qwen/Qwen2.5-0.5B-Instruct"
N, LARGO = 64, 96

PREGUNTAS = [
    "Por que el cielo se ve azul?", "Que es la fotosintesis?",
    "Como funciona un motor electrico?", "Que hace un compilador?",
    "Por que flota un barco de acero?", "Que es la presion atmosferica?",
    "Como se forma el granizo?", "Que es un numero primo?",
]


def main():
    prompts = [PREGUNTAS[i % len(PREGUNTAS)] for i in range(N)]
    tok = AutoTokenizer.from_pretrained(MODELO)
    # La plantilla se aplica una vez y se usa para las dos: si no, no estariamos
    # midiendo lo mismo
    textos = [tok.apply_chat_template([{"role": "user", "content": p}],
                                      tokenize=False, add_generation_prompt=True)
              for p in prompts]

    # transformers
    # en generación autorregresiva el modelo continúa desde el último token de la secuencia
    # si el relleno va a la derecha, el último token real es un <pad>
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    modelo = AutoModelForCausalLM.from_pretrained(
        MODELO, dtype=torch.float16, device_map="cuda")

    inicio = time.time()
    for i in range(0, N, 8):
        lote = tok(textos[i:i + 8], return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            modelo.generate(**lote, max_new_tokens=LARGO, do_sample=True,
                            temperature=0.8, pad_token_id=tok.pad_token_id)
    t_hf = time.time() - inicio
    print(f"transformers: {t_hf:6.1f} s", flush=True)

    # Hay que liberar la GPU antes de cargar vLLM: en una T4 (colab) no entran los dos
    # del solo quita la referencia de Python
    del modelo
    # el recolector tiene que correr para que se liberen los tensores
    gc.collect()
    # esto devuelve al driver los bloques que el caching allocator de PyTorch tenía reservados
    torch.cuda.empty_cache()

    # vLLM
    from vllm import LLM, SamplingParams

    # gpu_memory_utilization=0.75: vLLM pre-reserva ese porcentaje de la VRAM al
    #                              arrancar y administra él mismo esa región, por
    #                              eso hay que liberar lo anterior primero
    # max_model_len=640: contexto máximo por request
    # enforce_eager=True: desactiva los grafos de CUDA y la compilación con torch.compile,
    #                     arranca más rápido y es más robusto en una T4 compartida, a costa de velocidad
    llm = LLM(model=MODELO, dtype="half", gpu_memory_utilization=0.75,
              max_model_len=640, enforce_eager=True)

    inicio = time.time()
    salidas = llm.generate(textos, SamplingParams(max_tokens=LARGO, temperature=0.8))
    t_vllm = time.time() - inicio
    print(f"vLLM: {t_vllm:6.1f} s")
    print(f"\nvLLM es {t_hf / t_vllm:.1f} veces mas rapido para las mismas {N} generaciones")
    print(f"\nEjemplo de salida:\n  {salidas[0].outputs[0].text.strip()[:200]}")


# vLLM crea procesos auxiliares que vuelven a ejecutar este archivo desde el inicio.
# Esta condición asegura que el entrenamiento se ejecute solo en el proceso principal, lo que evita duplicaciones y errores al arrancar
if __name__ == "__main__":
    main()

Writing comparar.py


In [ ]:
!PYTHONUNBUFFERED=1 python comparar.py

config.json: 100% 659/659 [00:00<00:00, 2.36MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 8.37MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 55.3MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 79.9MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 86.8MB/s]

model.safetensors: downloading bytes:   4% 42.7M/988M [00:00<00:13, 68.0MB/s, 1.94MB/s  ]
model.safetensors: downloading bytes:  15% 145M/988M [00:01<00:05, 156MB/s, 12.9MB/s  ]
model.safetensors: downloading bytes:  39% 387M/988M [00:02<00:04, 131MB/s, 32.3MB/s  ]
model.safetensors: downloading bytes:  41% 408M/988M [00:03<00:04, 120MB/s, 33.2MB/s  ]
model.safetensors: downloading bytes:  84% 834M/988M [00:04<00:00, 223MB/s, 63.8MB/s  ]
model.safetensors: reconstructing file:  61% 604M/988M [00:09<00:06, 59.4MB/s, 29.3MB/s  ]
model.safetensors: downloading bytes: 100% 854M/854M [00:09<00:00, 85.6MB/s, 64.8MB/s  ]
model.safetensors: reconstructing file: 100% 988M/988M [00:09<00:00, 99.1MB/s, 68.5MB/s  ]
Loading w

* en el proyecto cada paso de GRPO genera un grupo de respuestas por cada prompt del lote
* casi todo el tiempo de GPU del entrenamiento se va en generar, no en calcular gradientes

---
## **Parte B) La plantilla de chat**

Esta parte corre en CPU: solo usa el tokenizador, es la más corta y la más importante

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
pregunta = "Cuanto es 7 x 8?"

crudo = pregunta
con_plantilla = tok.apply_chat_template(
    [{"role": "user", "content": pregunta}],
    tokenize=False, add_generation_prompt=True)

print("TEXTO PLANO")
print(repr(crudo))
print(f"  {len(tok(crudo).input_ids)} tokens\n")

print("CON LA PLANTILLA DE CHAT")
print(repr(con_plantilla))
print(f"  {len(tok(con_plantilla).input_ids)} tokens")

TEXTO PLANO
'Cuanto es 7 x 8?'
  9 tokens

CON LA PLANTILLA DE CHAT
'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nCuanto es 7 x 8?<|im_end|>\n<|im_start|>assistant\n'
  38 tokens


* el endpoint /v1/chat/completions aplica la plantilla siempre
* el endpoint /v1/completions nunca lo hace

Si un entrenamiento usa texto plano y la evaluación pega contra el endpoint de chat, el modelo ve dos formatos distintos y la diferencia se mide como si fuera un resultado

---
## **Parte C) el servidor**

* vLLM expone la misma API que OpenAI
* el código que consume no sabe qué modelo hay detrás ni dónde corre
* hasta acá el modelo era un objeto en memoria del notebook, ahora es un proceso separado con una API HTTP

In [ ]:
import subprocess
import time

import requests

MODELO = "Qwen/Qwen2.5-0.5B-Instruct"
URL = "http://127.0.0.1:8000"
LOG = "/content/vllm.log"

log = open(LOG, "w")
servidor = subprocess.Popen(
    ["vllm", "serve", MODELO,
     "--host", "127.0.0.1", "--port", "8000",
     "--dtype", "half",
     "--gpu-memory-utilization", "0.80",
     "--max-model-len", "1024"],
    stdout=log, stderr=subprocess.STDOUT,
)


def esperar(proceso, segundos=600):
    """
    - espera a /health, y corta si el proceso murio antes
    - sin esta segunda condicion, un servidor que nunca levanto se ve igual que
    uno lento, el error aparece cien lineas despues como un timeout
    """
    limite = time.time() + segundos
    while time.time() < limite:
        if proceso.poll() is not None:
            cola = open(LOG, encoding="utf-8").read()[-2500:]
            raise RuntimeError(f"vLLM murio. Final del log:\n\n{cola}")
        try:
            if requests.get(f"{URL}/health", timeout=2).status_code == 200:
                return
        except requests.RequestException:
            pass
        time.sleep(2)
    raise TimeoutError("vLLM no respondio a tiempo")


esperar(servidor)
print("servidor listo")
print("modelos registrados:", [m["id"] for m in requests.get(f"{URL}/v1/models").json()["data"]])

servidor listo
modelos registrados: ['Qwen/Qwen2.5-0.5B-Instruct']


In [ ]:
respuesta = requests.post(f"{URL}/v1/chat/completions", json={
    "model": MODELO,
    "messages": [{"role": "user", "content": "Explica en dos oraciones que es una GPU."}],
    "max_tokens": 120,
    "temperature": 0.0,
}, timeout=120).json()

print(respuesta["choices"][0]["message"]["content"].strip())
# El campo usage devuelve prompt_tokens: 41, esos 41 son la pregunta ya envuelta en la plantilla
print("\ntokens:", respuesta["usage"])

Una GPU (Graphics Processing Unit) es un dispositivo especializado diseñado para procesar y generar imágenes gráficas. Es capaz de realizar cálculos complejos rápidamente, lo que permite crear imágenes con alta calidad y velocidad. Las GPUs son especialmente útiles en aplicaciones como juegos de video, animación 3D, y análisis de datos grandes. Además, las GPUs pueden ser utilizadas para procesar datos de computación intensiva, como la generación de videos de alta resolución o el análisis de datos de larga duración.

tokens: {'prompt_tokens': 41, 'total_tokens': 156, 'completion_tokens': 115, 'prompt_tokens_details': None}


Vale pegarle también al endpoint crudo, para ver la diferencia de la parte B: el
mismo texto, sin plantilla, produce otra cosa.

In [ ]:
crudo = requests.post(f"{URL}/v1/completions", json={
    "model": MODELO,
    "prompt": "Explica en dos oraciones que es una GPU.",
    "max_tokens": 120,
    "temperature": 0.0,
}, timeout=120).json()

print("SIN PLANTILLA (/v1/completions)")
print(crudo["choices"][0]["text"].strip()[:400])

SIN PLANTILLA (/v1/completions)
Una GPU es un dispositivo especializado en procesar y almacenar datos, especialmente para el procesamiento de imágenes y videos. Es conocida por su capacidad de procesar grandes cantidades de datos en un tiempo muy corto, lo cual permite a los usuarios tener acceso a información rápida y precisa. Además, la GPU puede realizar muchas tareas simultáneamente, lo que hace que sea útil para aplicacione


---
## **Parte D) Base y adaptador, compuestos al servir**

* primero hace falta un adaptador, entrenaremos uno mínimo con peft
* El objetivo es trivial, que toda respuesta arranque con "Respuesta:", porque lo que importa no es qué aprendió, sino cómo se sirve

Antes hay que apagar el servidor: en la T4 no entran las dos cosas

In [ ]:
servidor.terminate()
servidor.wait(timeout=60)
print("servidor apagado")

servidor apagado


In [ ]:
%%writefile sft_corto.py
"""un adaptador LoRA minimo, para tener algo que servir"""

import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO = "Qwen/Qwen2.5-0.5B-Instruct"
DESTINO = "/content/adaptador_estilo"

EJEMPLOS = [
    ("Cuanto es 7 x 8?", "Respuesta: 56."),
    ("Capital de Francia?", "Respuesta: Paris."),
    ("Cuantos lados tiene un hexagono?", "Respuesta: seis."),
    ("Que planeta es el mas grande?", "Respuesta: Jupiter."),
    ("Cuanto es 15 + 27?", "Respuesta: 42."),
    ("Capital de Japon?", "Respuesta: Tokio."),
    ("Cuantos minutos tiene una hora?", "Respuesta: sesenta."),
    ("Que gas respiramos?", "Respuesta: oxigeno."),
    ("Cuanto es 9 x 9?", "Respuesta: 81."),
    ("Capital de Bolivia?", "Respuesta: Sucre."),
    ("Cuantos continentes hay?", "Respuesta: seis."),
    ("Que animal pone huevos y da leche?", "Respuesta: el ornitorrinco."),
]

tok = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).cuda()
modelo = get_peft_model(modelo, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM"))
modelo.print_trainable_parameters()

opt = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=2e-4)
modelo.train()

for epoca in range(8):
    perdida_total = 0.0
    for pregunta, respuesta in EJEMPLOS:
        texto = tok.apply_chat_template(
            [{"role": "user", "content": pregunta}],
            tokenize=False, add_generation_prompt=True) + respuesta + tok.eos_token
        ids = tok(texto, return_tensors="pt").input_ids.cuda()
        salida = modelo(input_ids=ids, labels=ids)
        salida.loss.backward()
        opt.step()
        opt.zero_grad()
        perdida_total += salida.loss.item()
    print(f"epoca {epoca + 1}/8  perdida media {perdida_total / len(EJEMPLOS):.3f}")

modelo.save_pretrained(DESTINO)
print(f"\nguardado en {DESTINO}")

Writing sft_corto.py


In [ ]:
!PYTHONUNBUFFERED=1 python sft_corto.py && ls -la /content/adaptador_estilo

Loading weights: 100% 290/290 [00:00<00:00, 380.40it/s]
trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184
epoca 1/8  perdida media 2.957
epoca 2/8  perdida media 1.064
epoca 3/8  perdida media 0.529
epoca 4/8  perdida media 0.322
epoca 5/8  perdida media 0.233
epoca 6/8  perdida media 0.170
epoca 7/8  perdida media 0.135
epoca 8/8  perdida media 0.119

guardado en /content/adaptador_estilo
total 4272
drwxr-xr-x 2 root root    4096 Oct  6 08:33 .
drwxr-xr-x 1 root root    4096 Oct  6 08:33 ..
-rw-r--r-- 1 root root    1132 Oct  6 08:33 adapter_config.json
-rw------- 1 root root 4350392 Oct  6 08:33 adapter_model.safetensors
-rw-r--r-- 1 root root    5206 Oct  6 08:33 README.md


Esos dos archivos son todo el artefacto:
* adapter_config.json con el rango y los módulos
* adapter_model.safetensors con las matrices A y B

Ahora se levanta el servidor con los dos: la base y el adaptador registrado con un nombre

In [ ]:
log = open(LOG, "w")
servidor = subprocess.Popen(
    ["vllm", "serve", MODELO,
     "--host", "127.0.0.1", "--port", "8000",
     "--dtype", "half",
     "--gpu-memory-utilization", "0.80",
     "--max-model-len", "1024",
     "--enable-lora",
     # Hay que declararlo, el valor por defecto coincide justo con r=16 y un
     # adaptador de rango mayor se rechaza al cargarse, no al arrancar.
     "--max-lora-rank", "16",
     "--max-loras", "1",
     "--lora-modules", "estilo=/content/adaptador_estilo"],
    stdout=log, stderr=subprocess.STDOUT,
)
esperar(servidor)

print("modelos registrados:", [m["id"] for m in requests.get(f"{URL}/v1/models").json()["data"]])

modelos registrados: ['Qwen/Qwen2.5-0.5B-Instruct', 'estilo']


**Dos modelos en un servidor, con una sola copia de los pesos base en memoria**
Cambiar de uno al otro es cambiar el campo `model` del pedido.

In [ ]:
PRUEBAS = ["Cuanto es 12 x 12?", "Capital de Italia?", "Cuantos dias tiene febrero?"]


def preguntar(modelo, pregunta):
    r = requests.post(f"{URL}/v1/chat/completions", json={
        "model": modelo,
        "messages": [{"role": "user", "content": pregunta}],
        "max_tokens": 60, "temperature": 0.0,
    }, timeout=120).json()
    return r["choices"][0]["message"]["content"].strip()


for pregunta in PRUEBAS:
    print(f"\n{'=' * 70}\n{pregunta}")
    print(f"\n  base     {preguntar(MODELO, pregunta)}")
    print(f"\n  estilo   {preguntar('estilo', pregunta)}")


Cuanto es 12 x 12?

  base     12 x 12 = 144

  estilo   Respuesta: 144.

Capital de Italia?

  base     La capital di Italia è Roma. È la città più grande e popolare del paese, con una popolazione di circa 4,700.000 residenti (2019). Roma è stata fondata nel 595 a.C., durante il periodo

  estilo   Respuesta: Roma.

Cuantos dias tiene febrero?

  base     El número de días en febrero varía dependiendo del calendario que se utilice:

1. Si el año es bisiesto (es decir, un año que cumple con la regla 370), hay 28 días.

2. Si el año no es b

  estilo   Respuesta: 27.


In [ ]:
servidor.terminate()
servidor.wait(timeout=60)
print("servidor apagado")

servidor apagado


---
**Conclusiones**

- vLLM y transformers generan lo mismo; la diferencia es de rendimiento, y en un bucle de RL eso decide si la corrida es viable
- la plantilla de chat la aplica el endpoint, no el modelo
- entrenar con un formato y evaluar con otro produce números que parecen resultados
- servir significa un proceso con una API, no un objeto en memoria, lo que se mide tiene que salir de ahí
- el adaptador se compone en el momento del pedido, una copia de la base, varios
  adaptadores, y revertir es cambiar un nombre

Esos cuatro hechos son los tres roles que vLLM cumple en el proyecto: genera durante el entrenamiento, sirve el artefacto para evaluarlo, y sirve al juez